# ERP003950_fastp_q30_u40: набор истины после фильтрации → PCR1 → фрагментация → PCR2 → NovaSeq PE150

Ветка предобработки: `results/ERP003950_fastp_q30_u40/`.

Биологический набор истины берётся только из её собственного `post_annotation_filtered/airr_pass`.

В ERP003950 нет UMI, поэтому точные V…J-последовательности дедуплицируются внутри каждого sample, а начальная abundance по умолчанию равна `one_per_unique`. Наблюдаемая multiplicity сохраняется только для QC и анализа чувствительности.

**Фрагментация:** `uniform single random-cut`.


## Структура выходных данных

`results/ERP003950_fastp_q30_u40/simulated/`
`insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut/`

Внутри:
- `00_primary_truth/`
- `01_pcr1/`
- `02_fragmentation/`
- `03_pcr2/`
- `04_read_allocation/`
- `05_fastq_native/`
- `06_fastq_pe150/`
- `logs/`
- `qc/`

Для FastQC/MultiQC общий `qc.ipynb` использует:
`DATASET="ERP003950_fastp_q30_u40"`, `STAGE="simulated"` и имя этой ветки.


## 1. Окружение

In [ ]:
import os, sys, sysconfig, subprocess, time, gzip, csv, math, re, shutil, json, hashlib
from pathlib import Path
from collections import defaultdict
import numpy as np

_ENV_CANDIDATES = [
    os.environ.get("BCR_ENV", ""),
    os.environ.get("CONDA_PREFIX", ""),
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
    "/Users/epishkin/mamba/envs/bcr_env",
]
_CONDA_ENV = next(
    (p for p in _ENV_CANDIDATES if p and os.path.isdir(str(Path(p) / "bin"))),
    None,
)
if not _CONDA_ENV:
    raise FileNotFoundError("bcr_env not found; activate it or set BCR_ENV")

os.environ["PATH"] = str(Path(_CONDA_ENV) / "bin") + ":" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"

for _site in [
    str(Path(_CONDA_ENV) / "lib/python3.11/site-packages"),
    str(Path(_CONDA_ENV) / "lib/python3.12/site-packages"),
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)

for tool in ("iss",):
    path = shutil.which(tool)
    if not path:
        raise RuntimeError(f"Required tool not found: {tool}")
    print(f"{tool}: {path}")

print("numpy:", np.__version__)


## 2. Параметры

In [ ]:
BRANCH_NAME = "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut"
SOURCE_STAGE = "post_annotation_filtered"

RESULT_DATASET = "ERP003950_fastp_q30_u40"
RAW_DATASET = "ERP003950"

SAMPLES = [
    "ERR346596",
    "ERR346597",
    "ERR346598",
    "ERR346599",
    "ERR346600",
    "ERR346601",
]

RUN_LOCUS = {sample: "IGH" for sample in SAMPLES}

def resolve_volume():
    candidates = []

    env_root = os.environ.get("BCR_VOLUME")
    if env_root:
        candidates.append(Path(env_root))

    candidates.extend([
        Path("/data/user/epishkin"),
        Path("/Users/epishkin/workspace/bcr-assembler"),
    ])

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])

    seen = set()

    for root in candidates:
        key = str(root)
        if key in seen:
            continue
        seen.add(key)

        if (
            (root / "results" / RESULT_DATASET).is_dir()
            and (root / "raw" / RAW_DATASET).is_dir()
        ):
            return root

    raise FileNotFoundError(
        f"Cannot locate results/{RESULT_DATASET} together with raw/{RAW_DATASET}. "
        "Set BCR_VOLUME explicitly."
    )

VOLUME = resolve_volume()
DATASET_DIR = VOLUME / "results" / RESULT_DATASET

POST_FILTER_DIR = DATASET_DIR / SOURCE_STAGE
TRUTH_AIRR_DIR = POST_FILTER_DIR / "airr_pass"
FILTERED_FASTQ_DIR = POST_FILTER_DIR / "fastq"
FILTER_SUMMARY_PATH = POST_FILTER_DIR / "filter_summary.json"

RAW_FASTQ_DIR = VOLUME / "raw" / RAW_DATASET

if not FILTER_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"Post-annotation filter summary not found: {FILTER_SUMMARY_PATH}. "
        "Run the fastp_q30_u40 mouse post-annotation filter first."
    )

FILTER_SUMMARY = json.loads(FILTER_SUMMARY_PATH.read_text())
FILTER_RULES = FILTER_SUMMARY["filters"]

assert FILTER_SUMMARY.get("dataset") == RESULT_DATASET
assert FILTER_RULES["require_expected_locus"] == RUN_LOCUS

MIN_TEMPLATE_LENGTH = int(FILTER_RULES["min_vj_span_nt"])
MIN_V_IDENTITY = float(FILTER_RULES["min_v_identity_percent"])
MIN_J_IDENTITY = float(FILTER_RULES["min_j_identity_percent"])
MAX_V_SUPPORT = float(FILTER_RULES["max_v_support_evalue"])
MAX_J_SUPPORT = float(FILTER_RULES["max_j_support_evalue"])

TARGET_READ_LENGTH = 150

OUT_BASE = DATASET_DIR / "simulated" / BRANCH_NAME
TRUTH_DIR = OUT_BASE / "00_primary_truth"
PCR1_DIR = OUT_BASE / "01_pcr1"
FRAGMENTATION_DIR = OUT_BASE / "02_fragmentation"
PCR2_DIR = OUT_BASE / "03_pcr2"
ALLOCATION_DIR = OUT_BASE / "04_read_allocation"
FASTQ_NATIVE_DIR = OUT_BASE / "05_fastq_native"
FASTQ_DIR = OUT_BASE / "06_fastq_pe150"
MODEL_DIR = OUT_BASE / "model"
LOGS_DIR = OUT_BASE / "logs"
QC_DIR = OUT_BASE / "qc"

TEMPLATES_DIR = TRUTH_DIR
COUNTS_DIR = ALLOCATION_DIR

for directory in (
    TRUTH_DIR,
    PCR1_DIR,
    FRAGMENTATION_DIR,
    PCR2_DIR,
    ALLOCATION_DIR,
    FASTQ_NATIVE_DIR,
    FASTQ_DIR,
    MODEL_DIR,
    LOGS_DIR,
    QC_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

NPROC = 8
SEED = 42
FORCE = False
CLEAN_OLD_OUTPUTS = False
COMPRESS = True

DROP_SEQUENCES_WITH_N = True
STARTING_COPIES_MODE = "one_per_unique"

PCR_CYCLES = 25
PCR_EFFICIENCY_MEAN = 0.85
PCR_EFFICIENCY_CONCENTRATION = 80.0
PCR_MAX_COPIES = 10**12

# Фрагментация и отбор библиотеки.
#
# Важное отличие от прежней реализации: число PCR1-копий НЕ обязано
# сохраняться как число фрагментов. Сначала моделируется конечный aliquot
# молекул из огромного PCR1-пула, затем каждая выбранная молекула физически
# разрезается. До size-selection проверяется сохранение нуклеотидной массы.
SEQUENCE_TYPE = "amplicon"
FRAGMENTATION_MODEL = "uniform_single_cut"

# Вычислительный library-input bottleneck: размер aliquot равен суммарному
# числу исходных молекул до PCR1, умноженному на этот коэффициент, но состав
# aliquot выбирается пропорционально abundance после PCR1. Так мы сохраняем
# PCR1-induced abundance без перебора триллионов копий.
LIBRARY_INPUT_SCALE = 1.0

# После разрезания выполняется отдельный мягкий size-selection. Параметры
# 200/40 теперь описывают отбор библиотеки, а не прямую генерацию длины.
SIZE_SELECTION_TARGET = 200
SIZE_SELECTION_SD = 40
MIN_SEQUENCABLE_FRAGMENT_LENGTH = 151

FRAGMENTATION_MODEL_PROVENANCE = {
    "model": FRAGMENTATION_MODEL,
    "library_input": "finite aliquot sampled from PCR1 pool proportional to PCR1 abundance",
    "library_input_scale_vs_starting_copies": LIBRARY_INPUT_SCALE,
    "breakpoints": "exactly one uniformly sampled inter-base bond per library-input molecule",
    "daughter_fragments": "both physical daughters are created before size selection",
    "size_selection": {
        "type": "Gaussian acceptance kernel",
        "target_nt": SIZE_SELECTION_TARGET,
        "sd_nt": SIZE_SELECTION_SD,
        "min_sequencable_nt": MIN_SEQUENCABLE_FRAGMENT_LENGTH,
    },
    "scientific_analogs": [
        "BEERS2: default sequence-independent uniform bond breaking + separate size selection; DOI 10.1093/bib/bbae164",
        "Flux Simulator: modular fragmentation + size selection; DOI 10.1093/nar/gks666",
    ],
}

LIBRARY_PCR_CYCLES = 10
LIBRARY_PCR_EFFICIENCY_MEAN = 0.85
LIBRARY_PCR_EFFICIENCY_CONCENTRATION = 80.0
LIBRARY_PCR_MAX_COPIES = 10**15

READ_BUDGET_MODE = "match_raw_pairs"
FIXED_READ_PAIRS = 500_000

SEQUENCING_ERROR_MODEL_PROVENANCE = {
    "type": "InSilicoSeq bundled NovaSeq profile",
    "conditioned_on_post_annotation_filter": False,
}

print("VOLUME:", VOLUME)
print("RESULT_DATASET:", RESULT_DATASET)
print("RAW_DATASET:", RAW_DATASET)
print("SOURCE STAGE:", SOURCE_STAGE)
print("TRUTH AIRR:", TRUTH_AIRR_DIR)
print("BRANCH:", BRANCH_NAME)
print("OUT_BASE:", OUT_BASE)
print("filter rules:", FILTER_RULES)


In [ ]:
# ---------------------------------------------------------------------
# Очистка существующих сгенерированных артефактов
# ---------------------------------------------------------------------
# Удаляться могут только файлы внутри этой ветки симуляции.
# Исходные raw, filtered и annotation данные не изменяются.

def clean_previous_generated_outputs():
    if not (FORCE and CLEAN_OLD_OUTPUTS):
        print("Cleanup disabled.")
        return

    generated_dirs = [
        TRUTH_DIR,
        PCR1_DIR,
        FRAGMENTATION_DIR,
        PCR2_DIR,
        ALLOCATION_DIR,
        FASTQ_NATIVE_DIR,
        FASTQ_DIR,
        MODEL_DIR,
        QC_DIR,
        LOGS_DIR,
    ]

    for directory in generated_dirs:
        if not directory.exists():
            continue
        for path in directory.iterdir():
            if path.is_file() or path.is_symlink():
                path.unlink()
            elif path.is_dir():
                shutil.rmtree(path)

    for directory in generated_dirs:
        directory.mkdir(parents=True, exist_ok=True)

    print(f"Cleaned previous generated outputs under: {OUT_BASE}")

clean_previous_generated_outputs()


## 3. Вспомогательные функции

In [ ]:
def open_text(path, mode="rt"):
    return gzip.open(path, mode) if str(path).endswith(".gz") else open(path, mode)

def iter_fastq(path):
    with open_text(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                return
            seq = h.readline().rstrip("\n\r")
            plus = h.readline()
            qual = h.readline()
            if not plus or not qual:
                raise ValueError(f"Truncated FASTQ: {path}")
            yield seq

def count_fastq(path):
    return sum(1 for _ in iter_fastq(path))

def iter_fasta(path):
    with open(path) as h:
        name, chunks = None, []
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:], []
            else:
                chunks.append(line)
        if name is not None:
            yield name, "".join(chunks)

def run_with_heartbeat(cmd, log_path, heartbeat=30, shell=False):
    t0 = time.time()
    with open(log_path, "w") as log_h:
        proc = subprocess.Popen(cmd, stdout=log_h, stderr=subprocess.STDOUT, text=True, shell=shell)
        label = cmd if shell else " ".join(map(str, cmd))
        print(f"[run] {label}\n  pid={proc.pid} log={log_path}")
        while proc.poll() is None:
            print(f"  running: elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}); see {log_path}")
    print(f"done in {(time.time()-t0)/60:.1f} min")

def sample_seed(sample, extra=0):
    return int(SEED + extra + sum((i + 1) * ord(ch) for i, ch in enumerate(sample)))

## 4. Построение точного набора истины V…J после фильтрации

`post_annotation_filtered/airr_pass` является источником данных.

Для каждого sample ноутбук:
1. повторно проверяет условия фильтра с помощью assertions;
2. обрезает наблюдаемую нуклеотидную последовательность до `v_sequence_start:j_sequence_end`;
3. исключает последовательности с неоднозначными N только по правилу симуляции;
4. схлопывает точные V…J-последовательности внутри sample;
5. сохраняет наблюдаемую multiplicity для QC.

Поскольку в ERP003950 нет UMI, начальная abundance для PCR по умолчанию равна одной
молекуле на каждый точный уникальный V…J template.


In [ ]:
def _truthy(x):
    return str(x).strip().lower() in {"t", "true", "1", "yes"}

def _falsey(x):
    return str(x).strip().lower() in {"f", "false", "0", "no"}

def _as_int(x):
    try:
        return int(float(x))
    except (TypeError, ValueError):
        return None

def _as_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

FILTER_SUMMARY_SHA256 = _sha256(FILTER_SUMMARY_PATH)
PROVENANCE_PATH = TRUTH_DIR / "source_provenance.json"

def _validate_post_filter_row(row, sample):
    problems = []

    v_start = _as_int(row.get("v_sequence_start"))
    j_end = _as_int(row.get("j_sequence_end"))
    v_germline_start = _as_int(row.get("v_germline_start"))
    v_identity = _as_float(row.get("v_identity"))
    j_identity = _as_float(row.get("j_identity"))
    v_support = _as_float(row.get("v_support"))
    j_support = _as_float(row.get("j_support"))

    if (
        v_start is None
        or j_end is None
        or j_end - v_start + 1 < MIN_TEMPLATE_LENGTH
    ):
        problems.append("V-J span")

    if v_germline_start != 1:
        problems.append("V 5-prime completeness")

    if not _truthy(row.get("complete_vdj")):
        problems.append("complete_vdj")

    if str(row.get("locus", "")).strip() != RUN_LOCUS[sample]:
        problems.append("locus")

    if not str(row.get("v_call", "")).strip():
        problems.append("v_call")
    if not str(row.get("j_call", "")).strip():
        problems.append("j_call")

    if not _truthy(row.get("productive")):
        problems.append("productive")

    if not _falsey(row.get("stop_codon")):
        problems.append("stop_codon")

    if v_identity is None or v_identity < MIN_V_IDENTITY:
        problems.append("v_identity")
    if j_identity is None or j_identity < MIN_J_IDENTITY:
        problems.append("j_identity")
    if v_support is None or v_support > MAX_V_SUPPORT:
        problems.append("v_support")
    if j_support is None or j_support > MAX_J_SUPPORT:
        problems.append("j_support")

    if problems:
        raise RuntimeError(
            f"{sample} airr_pass violates filter_summary contract for "
            f"{row.get('sequence_id')}: {', '.join(problems)}"
        )

def _current_provenance():
    return {
        "dataset": RESULT_DATASET,
        "source_stage": SOURCE_STAGE,
        "truth_airr_dir": str(TRUTH_AIRR_DIR),
        "filtered_fastq_dir": str(FILTERED_FASTQ_DIR),
        "filter_summary": str(FILTER_SUMMARY_PATH),
        "filter_summary_sha256": FILTER_SUMMARY_SHA256,
        "filter_rules": FILTER_RULES,
        "simulation_specific_truth_rules": {
            "drop_sequences_with_N": DROP_SEQUENCES_WITH_N,
            "crop_to_annotated_V_to_J": True,
            "collapse_scope": "within_sample",
            "collapse_key": ["VJ_sequence"],
            "starting_copies_mode": STARTING_COPIES_MODE,
            "dataset_has_umi": False,
        },
        "fragmentation_model": FRAGMENTATION_MODEL_PROVENANCE,
        "sequencing_error_model": SEQUENCING_ERROR_MODEL_PROVENANCE,
        "branch": BRANCH_NAME,
        "seed": SEED,
    }

def _check_cached_truth_provenance():
    if not PROVENANCE_PATH.exists():
        return False

    old = json.loads(PROVENANCE_PATH.read_text())
    if old.get("filter_summary_sha256") != FILTER_SUMMARY_SHA256:
        raise RuntimeError(
            "post_annotation_filtered/filter_summary.json changed since this "
            "simulation truth was built. Use FORCE=True and "
            "CLEAN_OLD_OUTPUTS=True, or choose a new branch."
        )
    return True

def build_sample_templates(sample, force=FORCE):
    out_fasta = TRUTH_DIR / f"{sample}_templates.fasta"
    out_tsv = TRUTH_DIR / f"{sample}_template_qc.tsv"
    audit_path = QC_DIR / f"{sample}_source_truth_audit.tsv"

    if (
        out_fasta.exists()
        and out_tsv.exists()
        and audit_path.exists()
        and not force
    ):
        _check_cached_truth_provenance()
        print(f"[{sample}] [skip] post-filter truth exists")
        return

    airr = TRUTH_AIRR_DIR / f"{sample}.airr.tsv"
    filtered_fastq = FILTERED_FASTQ_DIR / f"{sample}_filtered.fastq.gz"

    if not airr.exists():
        raise FileNotFoundError(f"Filtered AIRR missing: {airr}")
    if not filtered_fastq.exists():
        raise FileNotFoundError(f"Filtered FASTQ missing: {filtered_fastq}")

    expected_passed = int(FILTER_SUMMARY["samples"][sample]["passed"])

    counts = defaultdict(int)
    observed = defaultdict(int)

    with open(airr, newline="") as handle:
        reader = csv.DictReader(handle, delimiter="\t")

        for row in reader:
            counts["airr_pass_rows"] += 1
            _validate_post_filter_row(row, sample)

            start = int(row["v_sequence_start"]) - 1
            end = int(row["j_sequence_end"])
            seq = (row.get("sequence") or "")[start:end].upper()

            if len(seq) < MIN_TEMPLATE_LENGTH:
                raise RuntimeError(
                    f"{sample}: cropped V-J sequence shorter than filter threshold "
                    f"for {row.get('sequence_id')}"
                )

            if DROP_SEQUENCES_WITH_N and "N" in seq:
                counts["simulation_excluded_N"] += 1
                continue

            observed[seq] += 1
            counts["simulation_eligible_rows"] += 1

    if counts["airr_pass_rows"] != expected_passed:
        raise RuntimeError(
            f"{sample}: airr_pass rows={counts['airr_pass_rows']:,}, "
            f"but filter_summary passed={expected_passed:,}"
        )

    ordered = sorted(
        observed.items(),
        key=lambda item: (-item[1], item[0]),
    )

    with open(out_fasta, "w") as fasta, open(out_tsv, "w", newline="") as table:
        fields = [
            "template_id",
            "locus",
            "length",
            "observed_multiplicity",
        ]
        writer = csv.DictWriter(table, fieldnames=fields, delimiter="\t")
        writer.writeheader()

        for i, (seq, multiplicity) in enumerate(ordered, 1):
            template_id = f"{sample}_IGH_tpl_{i:07d}"
            fasta.write(f">{template_id}\n{seq}\n")
            writer.writerow({
                "template_id": template_id,
                "locus": "IGH",
                "length": len(seq),
                "observed_multiplicity": multiplicity,
            })

    audit = {
        "sample": sample,
        "expected_passed_from_filter_summary": expected_passed,
        "airr_pass_rows": counts["airr_pass_rows"],
        "simulation_eligible_rows": counts["simulation_eligible_rows"],
        "simulation_excluded_N": counts["simulation_excluded_N"],
        "unique_exact_vj_templates": len(ordered),
    }
    with open(audit_path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(audit), delimiter="\t")
        writer.writeheader()
        writer.writerow(audit)

    print(
        f"[{sample}] airr_pass={counts['airr_pass_rows']:,}; "
        f"simulation_eligible={counts['simulation_eligible_rows']:,}; "
        f"unique_VJ={len(ordered):,}; excluded_N={counts['simulation_excluded_N']:,}"
    )

for sample in SAMPLES:
    build_sample_templates(sample)

PROVENANCE_PATH.write_text(
    json.dumps(_current_provenance(), indent=2) + "\n"
)
print("provenance:", PROVENANCE_PATH)


## 5. Сводка QC точного набора истины V…J


In [ ]:
template_summary = []

for sample in SAMPLES:
    path = TRUTH_DIR / f"{sample}_template_qc.tsv"
    rows = list(csv.DictReader(open(path), delimiter="\t"))
    multiplicities = np.asarray(
        [int(row["observed_multiplicity"]) for row in rows],
        dtype=np.int64,
    )

    template_summary.append({
        "sample": sample,
        "locus": "IGH",
        "unique_templates": len(rows),
        "filtered_reads_represented": int(multiplicities.sum()),
        "singleton_templates": int((multiplicities == 1).sum()),
        "max_observed_multiplicity": (
            int(multiplicities.max()) if len(multiplicities) else 0
        ),
    })

out = QC_DIR / "template_summary.tsv"
with open(out, "w", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=list(template_summary[0]),
        delimiter="\t",
    )
    writer.writeheader()
    writer.writerows(template_summary)

print(*template_summary, sep="\n")
print("wrote", out)


## 6. Встроенная модель ошибок NovaSeq из InSilicoSeq

Эта ветка не обучает эмпирическую модель ошибок на ERP003950. Используется общий
встроенный профиль NovaSeq, что делает слой ошибок секвенирования сопоставимым между датасетами.


In [ ]:
import iss
ISS_PROFILE_NAME="NovaSeq"; ISS_NATIVE_READ_LENGTH=151
profile_path=Path(iss.__file__).resolve().parent/"profiles"/ISS_PROFILE_NAME
with np.load(profile_path,allow_pickle=True) as z:
    assert int(z["read_length"])==ISS_NATIVE_READ_LENGTH
    print({"profile":ISS_PROFILE_NAME,"model":str(z["model"].item()),"native_read_length":int(z["read_length"]),"target":TARGET_READ_LENGTH})


In [ ]:
# В этой ветке модель не обучается: используется встроенный KDE-профиль NovaSeq.

In [ ]:
# Происхождение модели записывается в ячейке параметров выше.

## 7. Бюджет секвенирования для каждого sample


In [ ]:
def get_read_budget(sample):
    if READ_BUDGET_MODE == "fixed":
        return int(FIXED_READ_PAIRS)

    if READ_BUDGET_MODE == "match_raw_pairs":
        r1 = RAW_FASTQ_DIR / f"{sample}_1.fastq.gz"
        r2 = RAW_FASTQ_DIR / f"{sample}_2.fastq.gz"

        if not r1.exists() or not r2.exists():
            raise FileNotFoundError(
                f"Missing raw pair for {sample}: {r1}, {r2}"
            )

        n1 = count_fastq(r1)
        n2 = count_fastq(r2)

        if n1 != n2:
            raise ValueError(
                f"{sample}: R1/R2 count mismatch: {n1:,} vs {n2:,}"
            )

        return n1

    raise ValueError(f"Unknown READ_BUDGET_MODE={READ_BUDGET_MODE!r}")

READ_BUDGETS = {sample: get_read_budget(sample) for sample in SAMPLES}
print(READ_BUDGETS)


## 8. PCR1 для точных V…J templates после фильтрации


In [ ]:
def load_dedup_templates(sample):
    fasta = TRUTH_DIR / f"{sample}_templates.fasta"
    qc = TRUTH_DIR / f"{sample}_template_qc.tsv"

    meta = {
        row["template_id"]: row
        for row in csv.DictReader(open(qc), delimiter="\t")
    }

    rows = []
    for template_id, seq in iter_fasta(fasta):
        m = meta[template_id]
        rows.append({
            "template_id": template_id,
            "sequence": seq,
            "length": len(seq),
            "locus": m["locus"],
            "observed_multiplicity": int(m["observed_multiplicity"]),
        })

    return rows

def starting_copies(row):
    if STARTING_COPIES_MODE == "one_per_unique":
        return 1

    if STARTING_COPIES_MODE == "observed_multiplicity":
        return row["observed_multiplicity"]

    raise ValueError(STARTING_COPIES_MODE)

def branching_pcr(n0, efficiency, cycles, rng):
    n = int(n0)

    for _ in range(cycles):
        if n <= 0:
            return 0

        n += int(rng.binomial(n, efficiency))

        if n >= PCR_MAX_COPIES:
            return int(PCR_MAX_COPIES)

    return n

def simulate_pcr_pool(sample):
    rng = np.random.default_rng(sample_seed(sample, 1000))
    rows = load_dedup_templates(sample)

    mean = PCR_EFFICIENCY_MEAN
    concentration = PCR_EFFICIENCY_CONCENTRATION

    if not (0 < mean <= 1):
        raise ValueError("PCR_EFFICIENCY_MEAN must be in (0,1]")

    if mean == 1:
        alpha = beta = None
    else:
        alpha = mean * concentration
        beta = (1 - mean) * concentration

    for row in rows:
        efficiency = (
            1.0
            if mean == 1
            else float(rng.beta(alpha, beta))
        )

        n0 = starting_copies(row)
        n_pcr = branching_pcr(
            n0,
            efficiency,
            PCR_CYCLES,
            rng,
        )

        row.update(
            starting_copies=n0,
            pcr_efficiency=efficiency,
            pcr_copies=n_pcr,
        )

    return rows

_test_rng = np.random.default_rng(1)
assert branching_pcr(1, 1.0, 10, _test_rng) == 2**10
assert branching_pcr(1, 0.0, 10, _test_rng) == 1
print("branching PCR smoke tests: OK")


## 9. Фрагментация: baseline random-cut

Эта версия больше не создаёт фиксированное число «репрезентативных фрагментов» на шаблон.

1. Из огромного PCR1-пула выбирается конечный aliquot молекул пропорционально abundance после PCR1. Размер aliquot задаётся через `LIBRARY_INPUT_SCALE` относительно числа исходных молекул до PCR1.
2. Для каждой выбранной физической молекулы выбирается **ровно одна** межнуклеотидная связь с равной вероятностью среди всех `L-1` возможных позиций.
3. Молекула действительно делится на **два дочерних фрагмента**.
4. После физического разрыва выполняется отдельный мягкий size-selection вокруг 200 nt (`SD=40`).
5. В PCR2 переходят только фрагменты, прошедшие size-selection и достаточно длинные для используемого профиля секвенирования.

Математически это соответствует независимому uniform random cut каждой молекулы, но события агрегируются по одинаковым координатам разрыва, поэтому триллионы PCR1-копий не создаются как отдельные Python-объекты.

Научная мотивация: BEERS2 использует sequence-independent uniform bond breaking как базовый режим fragmentation и отделяет fragmentation от size selection (DOI `10.1093/bib/bbae164`). Flux Simulator также моделирует fragmentation и size selection как отдельные стадии (DOI `10.1093/nar/gks666`).


In [ ]:
def _size_selection_probability(length):
    """Probability that a physical fragment enters the PCR2 library."""
    length = int(length)
    if length < MIN_SEQUENCABLE_FRAGMENT_LENGTH:
        return 0.0
    z = (length - SIZE_SELECTION_TARGET) / SIZE_SELECTION_SD
    return float(math.exp(-0.5 * z * z))


def _allocate_library_input(rows_pcr1, rng):
    """Sample a finite aliquot from the enormous PCR1 pool.

    The aliquot size is tied to pre-PCR molecular diversity (starting_copies),
    while template probabilities are determined by PCR1 output abundance.
    This keeps the simulation finite without flattening PCR1 effects.
    """
    starting_total = sum(max(0, int(r["starting_copies"])) for r in rows_pcr1)
    pcr1_total = sum(max(0, int(r["pcr_copies"])) for r in rows_pcr1)
    if starting_total <= 0 or pcr1_total <= 0:
        raise RuntimeError("empty PCR1 pool")

    requested = max(1, int(round(starting_total * LIBRARY_INPUT_SCALE)))
    n_input = min(requested, pcr1_total)
    weights = np.asarray([max(0, int(r["pcr_copies"])) for r in rows_pcr1], dtype=float)
    probs = weights / weights.sum()
    allocations = rng.multinomial(n_input, probs)
    return allocations, int(n_input), int(pcr1_total), int(starting_total)


def enumerate_fragments(sample, rows_pcr1, rng):
    """Uniform one-cut fragmentation of physical library-input molecules.

    Each selected molecule receives exactly one uniformly distributed cut at
    an inter-base bond. Both daughter molecules are created. A separate soft
    size-selection determines which daughters proceed to PCR2.
    """
    input_alloc, library_input_molecules, pcr1_total, starting_total = _allocate_library_input(rows_pcr1, rng)

    terminal = defaultdict(int)  # (template_id, locus, tlen, start, end) -> physical molecules
    input_nt_mass = 0
    library_input_templates = 0

    for r, n_input in zip(rows_pcr1, input_alloc):
        n_input = int(n_input)
        if n_input <= 0:
            continue
        library_input_templates += 1
        seq = r["sequence"]
        tlen = int(r["length"])
        if tlen < 2:
            continue

        input_nt_mass += n_input * tlen

        # Equivalent to cutting each physical molecule independently, but
        # aggregated over the L-1 possible inter-base bonds.
        cut_counts = rng.multinomial(
            n_input,
            np.full(tlen - 1, 1.0 / (tlen - 1), dtype=float),
        )
        for cut0 in np.flatnonzero(cut_counts):
            n = int(cut_counts[cut0])
            cut = int(cut0) + 1
            terminal[(r["template_id"], r["locus"], tlen, 0, cut)] += n
            terminal[(r["template_id"], r["locus"], tlen, cut, tlen)] += n

    terminal_fragment_molecules = sum(terminal.values())
    terminal_nt_mass = sum((end - start) * n for (_, _, _, start, end), n in terminal.items())
    if terminal_nt_mass != input_nt_mass:
        raise AssertionError(
            f"fragmentation nucleotide-mass conservation failed: input={input_nt_mass}, terminal={terminal_nt_mass}"
        )

    fragments = []
    retained_fragment_molecules = 0
    discarded_fragment_molecules = 0

    seq_by_id = {r["template_id"]: r["sequence"] for r in rows_pcr1}
    for (template_id, locus, tlen, start, end), n in terminal.items():
        flen = end - start
        p_keep = _size_selection_probability(flen)
        kept = int(rng.binomial(int(n), p_keep)) if p_keep > 0 else 0
        discarded_fragment_molecules += int(n) - kept
        if kept <= 0:
            continue
        retained_fragment_molecules += kept
        fragments.append({
            "fragment_id": f"{template_id}_rc_{start}_{end}",
            "template_id": template_id,
            "locus": locus,
            "template_length": int(tlen),
            "fragment_start": int(start),
            "fragment_length": int(flen),
            "sequence": seq_by_id[template_id][start:end],
            "fragment_input_copies": int(kept),
        })

    if not fragments:
        raise RuntimeError(f"{sample}: size selection removed the entire fragmented library")

    summary = {
        "sample": sample,
        "fragmentation_model": FRAGMENTATION_MODEL,
        "pcr1_molecules_total": int(pcr1_total),
        "starting_copies_total": int(starting_total),
        "library_input_molecules": int(library_input_molecules),
        "library_input_templates": int(library_input_templates),
        "fragmentation_events": int(library_input_molecules),
        "terminal_fragment_molecules": int(terminal_fragment_molecules),
        "terminal_fragment_species": int(len(terminal)),
        "retained_fragment_molecules": int(retained_fragment_molecules),
        "retained_fragment_species": int(len(fragments)),
        "discarded_fragment_molecules": int(discarded_fragment_molecules),
        "input_nt_mass": int(input_nt_mass),
        "terminal_nt_mass": int(terminal_nt_mass),
        "fragmentation_nt_conserved": bool(input_nt_mass == terminal_nt_mass),
        "size_selection_target_nt": int(SIZE_SELECTION_TARGET),
        "size_selection_sd_nt": int(SIZE_SELECTION_SD),
        "min_sequencable_fragment_nt": int(MIN_SEQUENCABLE_FRAGMENT_LENGTH),
    }
    return fragments, summary


## 10. PCR2 для фрагментов

PCR2 применяется к сохранённым фрагментам и создаёт отдельную таблицу этапа.


In [ ]:
def branching_pcr_vectorized(n0, efficiency, cycles, max_copies, rng):
    n = np.asarray(n0, dtype=np.int64).copy()
    eff = np.asarray(efficiency, dtype=np.float64)
    for _ in range(cycles):
        active = n > 0
        if not active.any():
            break
        new = np.zeros_like(n)
        new[active] = rng.binomial(n[active], eff[active])
        n = n + new
        np.minimum(n, max_copies, out=n)
    return n


def run_pcr2_on_fragments(fragments, rng):
    mean = LIBRARY_PCR_EFFICIENCY_MEAN
    conc = LIBRARY_PCR_EFFICIENCY_CONCENTRATION
    if mean == 1:
        efficiency = np.ones(len(fragments))
    else:
        alpha, beta = mean * conc, (1 - mean) * conc
        efficiency = rng.beta(alpha, beta, size=len(fragments))

    n0 = np.array([f["fragment_input_copies"] for f in fragments], dtype=np.int64)
    copies = branching_pcr_vectorized(
        n0, efficiency, LIBRARY_PCR_CYCLES, LIBRARY_PCR_MAX_COPIES, rng
    )
    for f, eff, cp in zip(fragments, efficiency, copies):
        f["library_pcr_efficiency"] = float(eff)
        f["library_pcr2_copies"] = int(cp)
    return fragments

_test_rng = np.random.default_rng(1)
_vec = branching_pcr_vectorized(
    np.array([1]), np.array([1.0]), 10, 10**15, _test_rng
)
assert int(_vec[0]) == 2**10
print("vectorized branching PCR smoke test: OK")


## 11. Сохранение PCR1 → фрагментация → PCR2 → точное распределение ридов по samples


In [ ]:
def write_dict_rows(path, rows, fields):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", newline="") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fields,
            delimiter="\t",
        )
        writer.writeheader()
        writer.writerows(
            {field: row[field] for field in fields}
            for row in rows
        )
    tmp.replace(path)

def read_dict_rows(path):
    with open(path, newline="") as handle:
        return list(csv.DictReader(handle, delimiter="\t"))

def run_pcr1_stage(sample, force=FORCE):
    out = PCR1_DIR / f"{sample}_pcr1_pool.tsv"

    if out.exists() and not force:
        print("[skip] PCR1", out)
        return

    rows = simulate_pcr_pool(sample)
    fields = [
        "template_id",
        "locus",
        "length",
        "observed_multiplicity",
        "starting_copies",
        "pcr_efficiency",
        "pcr_copies",
    ]

    write_dict_rows(out, rows, fields)
    print("PCR1 rows:", len(rows), "->", out)

def run_fragmentation_stage(sample, force=FORCE):
    inp = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    out = FRAGMENTATION_DIR / f"{sample}_fragments.tsv"
    summary_out = FRAGMENTATION_DIR / f"{sample}_fragmentation_summary.tsv"

    if out.exists() and summary_out.exists() and not force:
        print("[skip] fragmentation", out)
        return

    rows = []
    for row in read_dict_rows(inp):
        rows.append({
            **row,
            "length": int(row["length"]),
            "starting_copies": int(row["starting_copies"]),
            "pcr_copies": int(row["pcr_copies"]),
            "sequence": None,
        })

    sequences = dict(iter_fasta(TRUTH_DIR / f"{sample}_templates.fasta"))
    for row in rows:
        row["sequence"] = sequences[row["template_id"]]

    fragments, frag_summary = enumerate_fragments(
        sample,
        rows,
        np.random.default_rng(sample_seed(sample, 2000)),
    )

    fields = [
        "fragment_id", "template_id", "locus", "template_length",
        "fragment_start", "fragment_length", "sequence", "fragment_input_copies",
    ]
    write_dict_rows(out, fragments, fields)
    write_dict_rows(summary_out, [frag_summary], list(frag_summary))

    print("fragmentation model:", FRAGMENTATION_MODEL)
    print("fragmentation rows:", len(fragments), "->", out)
    print("fragmentation summary:", frag_summary)


def run_pcr2_stage(sample, force=FORCE):
    inp = FRAGMENTATION_DIR / f"{sample}_fragments.tsv"
    out = PCR2_DIR / f"{sample}_pcr2_pool.tsv"

    if out.exists() and not force:
        print("[skip] PCR2", out)
        return

    fragments = read_dict_rows(inp)

    for fragment in fragments:
        for key in (
            "template_length",
            "fragment_start",
            "fragment_length",
            "fragment_input_copies",
        ):
            fragment[key] = int(fragment[key])

    fragments = run_pcr2_on_fragments(
        fragments,
        np.random.default_rng(sample_seed(sample, 3000)),
    )

    fields = [
        "fragment_id",
        "template_id",
        "locus",
        "template_length",
        "fragment_start",
        "fragment_length",
        "sequence",
        "fragment_input_copies",
        "library_pcr_efficiency",
        "library_pcr2_copies",
    ]

    write_dict_rows(out, fragments, fields)
    print("PCR2 rows:", len(fragments), "->", out)

def run_allocation_stage(sample, force=FORCE):
    inp = PCR2_DIR / f"{sample}_pcr2_pool.tsv"
    out_counts = ALLOCATION_DIR / f"{sample}_read_counts.tsv"
    out_fasta = ALLOCATION_DIR / f"{sample}_selected_fragments.fasta"
    out_qc = ALLOCATION_DIR / f"{sample}_allocation.tsv"

    if (
        out_counts.exists()
        and out_fasta.exists()
        and out_qc.exists()
        and not force
    ):
        print("[skip] allocation", sample)
        return

    fragments = read_dict_rows(inp)

    for fragment in fragments:
        for key in (
            "template_length",
            "fragment_start",
            "fragment_length",
            "fragment_input_copies",
            "library_pcr2_copies",
        ):
            fragment[key] = int(fragment[key])

        fragment["library_pcr_efficiency"] = float(
            fragment["library_pcr_efficiency"]
        )

    weights = np.asarray(
        [fragment["library_pcr2_copies"] for fragment in fragments],
        dtype=float,
    )

    if not len(weights) or not np.isfinite(weights.sum()) or weights.sum() <= 0:
        raise RuntimeError(f"{sample}: PCR2 pool is empty or invalid")

    target = int(READ_BUDGETS[sample])
    rng = np.random.default_rng(sample_seed(sample, 4000))
    allocated = rng.multinomial(
        target,
        weights / weights.sum(),
    )

    counts_tmp = Path(str(out_counts) + ".tmp")
    fasta_tmp = Path(str(out_fasta) + ".tmp")

    with open(counts_tmp, "w", newline="") as count_handle, \
         open(fasta_tmp, "w") as fasta_handle:

        writer = csv.writer(count_handle, delimiter="\t")

        for fragment, n_pairs in zip(fragments, allocated):
            fragment["simulated_read_pairs"] = int(n_pairs)

            if n_pairs:
                # ISS readcount_file ожидает общее число ридов, поэтому число PE-пар умножается на 2.
                writer.writerow([
                    fragment["fragment_id"],
                    int(n_pairs) * 2,
                ])
                fasta_handle.write(
                    f">{fragment['fragment_id']}\n"
                    f"{fragment['sequence']}\n"
                )

    counts_tmp.replace(out_counts)
    fasta_tmp.replace(out_fasta)

    fields = [
        "fragment_id",
        "template_id",
        "locus",
        "template_length",
        "fragment_start",
        "fragment_length",
        "fragment_input_copies",
        "library_pcr_efficiency",
        "library_pcr2_copies",
        "simulated_read_pairs",
    ]

    write_dict_rows(out_qc, fragments, fields)

    if int(allocated.sum()) != target:
        raise AssertionError(
            f"{sample}: allocated {allocated.sum()} != target {target}"
        )

    print(
        f"[{sample}] allocated={target:,} pairs; "
        f"fragments={len(fragments):,}; "
        f"fragments_with_reads={(allocated > 0).sum():,}"
    )

for sample in SAMPLES:
    run_pcr1_stage(sample)
    run_fragmentation_stage(sample)
    run_pcr2_stage(sample)
    run_allocation_stage(sample)


## 12. QC для PCR1, фрагментации и PCR2


In [ ]:
pcr_summary = []
for sample in SAMPLES:
    pcr1_rows = read_dict_rows(PCR1_DIR / f"{sample}_pcr1_pool.tsv")
    frag_rows = read_dict_rows(FRAGMENTATION_DIR / f"{sample}_fragments.tsv")
    frag_summary = read_dict_rows(FRAGMENTATION_DIR / f"{sample}_fragmentation_summary.tsv")[0]
    pcr2_rows = read_dict_rows(PCR2_DIR / f"{sample}_pcr2_pool.tsv")

    pcr1_n = len(pcr1_rows)
    pcr1_molecules = sum(int(r["pcr_copies"]) for r in pcr1_rows)
    fragment_input_molecules = sum(int(r["fragment_input_copies"]) for r in frag_rows)

    if str(frag_summary["fragmentation_nt_conserved"]).lower() not in {"true", "1"}:
        raise AssertionError("fragmentation nucleotide-mass conservation QC failed")
    if int(frag_summary["retained_fragment_molecules"]) != fragment_input_molecules:
        raise AssertionError("fragmentation retained-molecule accounting QC failed")

    copies2 = [int(r["library_pcr2_copies"]) for r in pcr2_rows]
    eff2 = [float(r["library_pcr_efficiency"]) for r in pcr2_rows]

    allocated_n = allocated_nonzero = allocated_total = 0
    with open(ALLOCATION_DIR / f"{sample}_allocation.tsv") as h:
        for r in csv.DictReader(h, delimiter="\t"):
            n = int(r["simulated_read_pairs"])
            allocated_n += 1
            allocated_total += n
            allocated_nonzero += int(n > 0)

    pcr_summary.append({
        "sample": sample,
        "fragmentation_model": frag_summary["fragmentation_model"],
        "pcr1_templates": pcr1_n,
        "pcr1_molecules": pcr1_molecules,
        "library_input_molecules": int(frag_summary["library_input_molecules"]),
        "fragmentation_events": int(frag_summary["fragmentation_events"]),
        "terminal_fragment_molecules": int(frag_summary["terminal_fragment_molecules"]),
        "terminal_fragment_species": int(frag_summary["terminal_fragment_species"]),
        "retained_fragment_molecules": fragment_input_molecules,
        "retained_fragment_species": len(frag_rows),
        "discarded_fragment_molecules": int(frag_summary["discarded_fragment_molecules"]),
        "input_nt_mass": int(frag_summary["input_nt_mass"]),
        "terminal_nt_mass": int(frag_summary["terminal_nt_mass"]),
        "fragmentation_nt_conserved": True,
        "pcr2_fragments": len(copies2),
        "target_read_pairs": allocated_total,
        "fragments_with_reads": allocated_nonzero,
        "fragment_sampling_dropout": 1 - allocated_nonzero / allocated_n,
        "mean_pcr2_efficiency": float(np.mean(eff2)),
        "median_pcr2_copies": float(np.median(copies2)),
    })

out = QC_DIR / "pcr1_fragmentation_pcr2_summary.tsv"
write_dict_rows(out, pcr_summary, list(pcr_summary[0]))
print(*pcr_summary, sep="\n")


## 13. Генерация ридов PE150 с профилем NovaSeq


In [ ]:
def crop_fastq(in_path,out_path,length):
    tmp=Path(str(out_path)+".tmp")
    with open_text(in_path,"rt") as src,gzip.open(tmp,"wt") as dst:
        n=0
        while True:
            h=src.readline()
            if not h: break
            s=src.readline().rstrip("\n\r"); p=src.readline(); q=src.readline().rstrip("\n\r")
            if len(s)<length or len(q)<length: raise ValueError(f"short read in {in_path}: {len(s)}")
            dst.write(h); dst.write(s[:length]+"\n"); dst.write(p); dst.write(q[:length]+"\n"); n+=1
    tmp.replace(out_path); return n

def run_iss_generate(sample,force=FORCE):
    fa=ALLOCATION_DIR/f"{sample}_selected_fragments.fasta"; counts=ALLOCATION_DIR/f"{sample}_read_counts.tsv"; native_prefix=FASTQ_NATIVE_DIR/sample; final_prefix=FASTQ_DIR/sample; ext=".fastq.gz"
    nr1=Path(str(native_prefix)+f"_R1{ext}"); nr2=Path(str(native_prefix)+f"_R2{ext}"); r1=Path(str(final_prefix)+f"_R1{ext}"); r2=Path(str(final_prefix)+f"_R2{ext}")
    if r1.exists() and r2.exists() and not force: print("[skip]",sample); return
    cmd=["iss","generate","--genomes",str(fa),"--readcount_file",str(counts),"--sequence_type",SEQUENCE_TYPE,"--model",ISS_PROFILE_NAME,"--cpus",str(NPROC),"--output",str(native_prefix),"--seed",str(sample_seed(sample,5000)),"--compress"]
    run_with_heartbeat(cmd,LOGS_DIR/f"{sample}_iss_novaseq.log")
    n1=crop_fastq(nr1,r1,TARGET_READ_LENGTH); n2=crop_fastq(nr2,r2,TARGET_READ_LENGTH); assert n1==n2==READ_BUDGETS[sample]
    print("NovaSeq native PE151 -> exact PE150:",n1,"pairs")
for sample in SAMPLES: run_iss_generate(sample)


## 14. Итоговая проверка


In [ ]:
final_rows=[]
for sample in SAMPLES:
    r1=FASTQ_DIR/f"{sample}_R1.fastq.gz"; r2=FASTQ_DIR/f"{sample}_R2.fastq.gz"; n1=count_fastq(r1); n2=count_fastq(r2)
    lengths1=sorted({len(s) for s in iter_fastq(r1)}); lengths2=sorted({len(s) for s in iter_fastq(r2)})
    final_rows.append({"branch":BRANCH_NAME,"sample":sample,"expected_pairs":READ_BUDGETS[sample],"R1_reads":n1,"R2_reads":n2,"R1_lengths":",".join(map(str,lengths1)),"R2_lengths":",".join(map(str,lengths2)),"valid":n1==n2==READ_BUDGETS[sample] and lengths1==lengths2==[TARGET_READ_LENGTH]})
out=QC_DIR/"final_qc.tsv"; write_dict_rows(out,final_rows,list(final_rows[0])); assert all(r["valid"] for r in final_rows); print(*final_rows,sep="\n")


## Интерпретация и ограничения

- Симуляция относится только к ветке предобработки `ERP003950_fastp_q30_u40`.
- Источник набора истины — собственный `post_annotation_filtered/airr_pass` этой ветки.
- В датасете нет UMI: abundance по умолчанию равна `one_per_unique`, а наблюдаемая multiplicity сохраняется только для анализа чувствительности.
- Точная наблюдаемая V…J-последовательность сохраняет настоящую SHM; germline-последовательность не подставляется.
- PCR1/PCR2 представлены стохастическими ветвящимися моделями; ошибки полимеразы, химеры и истощение реагентов явно не моделируются.
- Для ошибок секвенирования используется встроенная модель InSilicoSeq NovaSeq. Нативные риды PE151 детерминированно обрезаются до PE150.
- Бюджет секвенирования каждого sample по умолчанию берётся из исходных R1/R2 в `raw/ERP003950`, поскольку ветка fastp представляет альтернативную предобработку того же физического датасета.
- Фрагментация использует один равновероятный random cut на выбранную физическую молекулу; оба дочерних фрагмента создаются до size-selection.
- Между PCR1 и fragmentation явно моделируется конечный library-input aliquot; до size-selection проверяется сохранение суммарной нуклеотидной массы.
